In [1]:
%pip install torch

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: C:\Users\achah\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
%pip install torch torchvision

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: C:\Users\achah\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
# all the imports
import os
from PIL import Image
from torchvision.datasets import ImageFolder as ImageFolder
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as np 
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split #splitsen van data in labeled/unlabeled/test
from sklearn.ensemble import RandomForestClassifier #basismodel
from sklearn.metrics import accuracy_score #model evalueren 
import pandas as pd

Afmetingen aangepast Trainingset

In [4]:
class GefilterdeTumorDataset(Dataset):
    def __init__(self, root, min=256, transform=None): # min is minimale grens en root is hoofdmap
        # Haal de submappen op als klassen
        self.classes = [
            d for d in os.listdir(root)
            if os.path.isdir(os.path.join(root, d))
        ]

        self.images = []
        self.transform = transform

        for index, group in enumerate(self.classes):
            group_map = os.path.join(root, group)

            for filename in os.listdir(group_map):
                if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                    pad = os.path.join(group_map, filename)

                    # Controleer de afmetingen
                    with Image.open(pad) as img:
                        width, height = img.size

                    if width >= min and height >= min:
                        self.images.append((pad, index))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        pad, label = self.images[idx]
        img = Image.open(pad).convert('RGB')

        if self.transform:
            img = self.transform(img)

        return img, label


In [5]:
# Toepassing in jouw bestaande pipeline
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor()
])

In [6]:
dataset_test = GefilterdeTumorDataset(root='./Testing', min=256, transform=transform)
dataloader_test = DataLoader(dataset_test, batch_size=32, shuffle=True)

print(f"Totaal aantal gefilterde foto's ingeladen: {len(dataset_test)}")

Totaal aantal gefilterde foto's ingeladen: 212


In [7]:
%pip install -U label-studio

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: C:\Users\achah\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
dataset_train = GefilterdeTumorDataset(root='./Training', min=256, transform=transform)

print(f"Totaal aantal gefilterde foto's ingeladen: {len(dataset_train)}")

Totaal aantal gefilterde foto's ingeladen: 2559


Laden en splitsen van data (labels vs unlabeled)

In [20]:
import pandas as pd
import os

# 1. Lees het opgeschoonde CSV-bestand in
df_gelabeled = pd.read_csv("labeled_scans.csv")

# 2. Maak direct de koppeling (omdat de namen nu perfect kloppen)
# Let op: check even of de kolom met bestandsnamen nog steeds "image" heet
gelabelde_bestandsnaam = dict(zip(df_gelabeled["image"], df_gelabeled["choice"]))

# 3. Inladen en verdelen
x_labeled = []
y_labeled = []
x_unlabeled = []

print("Foto's aan het inladen en verdelen...")

for i in range(len(dataset_train)):
    scan, _ = dataset_train[i] 
    
    # Pak het pad en haal de pure bestandsnaam eruit (bijv. 'gg_18.jpg')
    pad = dataset_train.images[i][0] 
    bestandsnaam = os.path.basename(pad) 

    # Checken en verdelen
    if bestandsnaam in gelabelde_bestandsnaam:
        x_labeled.append(scan)
        y_labeled.append(gelabelde_bestandsnaam[bestandsnaam])
    else:
        x_unlabeled.append(scan)

print(f"Klaar! {len(x_labeled)} gelabelde foto's en {len(x_unlabeled)} ongelabelde foto's ingeladen.")

Foto's aan het inladen en verdelen...
Klaar! 142 gelabelde foto's en 2417 ongelabelde foto's ingeladen.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Zet jouw tekst-labels eerst om naar getallen
label_naar_getal = {"Sagittaal": 0, "Axiaal": 1, "Coronaal": 2} # Check even of deze namen kloppen met je CSV
y_labeled_num = [label_naar_getal[label] for label in y_labeled]

# 2. De PyTorch variant van jouw pseudo-labeling functie
def pytorch_pseudo_labeling(x_labeled, y_labeled_num, x_unlabeled, model, threshold=0.95, max_iters=5):
    # We gebruiken een standaard Adam optimizer en CrossEntropy (ideaal voor classificatie)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    for iteration in range(max_iters):
        print(f"\n--- Iteratie {iteration + 1} ---")
        
        # --- STAP A: Trainen op gelabelde data ---
        model.train()
        # Omdat je geen DataLoaders gebruikt, stapelen we de losse foto's tot één 'batch'
        X_train = torch.stack(x_labeled)
        Y_train = torch.tensor(y_labeled_num)
        
        # We trainen het model even 5 rondes (epochs) per iteratie
        for epoch in range(5):
            optimizer.zero_grad()
            outputs = model(X_train)
            loss = criterion(outputs, Y_train)
            loss.backward()
            optimizer.step()
            
        print("Basismodel succesvol getraind/geüpdatet.")
        
        # --- STAP B: Voorspellen op ongelabelde data ---
        model.eval()
        nieuwe_x_labeled = []
        nieuwe_y_labeled = []
        overgebleven_x_unlabeled = []
        
        with torch.no_grad():
            for img in x_unlabeled:
                # We sturen ze 1 voor 1 door het model. img.unsqueeze(0) maakt er een 'batch van 1' van.
                output = model(img.unsqueeze(0)) 
                
                # Bereken de kansen (softmax) in percentages
                kansen = torch.softmax(output, dim=1)
                zekerheid = torch.max(kansen).item()
                voorspelling = torch.argmax(kansen).item()
                
                # --- STAP C: Selecteer hoge zekerheid ---
                if zekerheid >= threshold:
                    nieuwe_x_labeled.append(img)
                    nieuwe_y_labeled.append(voorspelling)
                else:
                    overgebleven_x_unlabeled.append(img)
        
        if len(nieuwe_x_labeled) == 0:
            print("Geen nieuwe beelden met >95% zekerheid gevonden. Pseudo-labeling stopt.")
            break
            
        # --- STAP D: Samenvoegen met originele data ---
        # .extend() voegt de nieuwe lijsten achteraan jouw bestaande lijsten toe
        x_labeled.extend(nieuwe_x_labeled)
        y_labeled_num.extend(nieuwe_y_labeled)
        
        # De ongelabelde bak bestaat nu alleen nog uit de overgebleven moeilijke beelden
        x_unlabeled = overgebleven_x_unlabeled
        
        print(f"{len(nieuwe_x_labeled)} pseudo-labels toegevoegd!")
        print(f"Nieuwe totale trainingsset: {len(x_labeled)} foto's.")
        
    return model


_IncompleteInputError: incomplete input (1145540459.py, line 74)